# OpenMontage trên Colab (T4)

Repo: https://github.com/calesthio/OpenMontage (giấy phép **AGPL-3.0** theo README).

**Nói thẳng về giới hạn (theo README đã đọc):**
- OpenMontage vận hành qua một **trợ lý lập trình AI** (Claude Code, Cursor, Codex...). README **không** nêu lệnh CLI riêng để render từ một tệp kịch bản. Luồng chung: `research -> proposal -> script -> scene_plan -> assets -> edit -> compose`, các bước sáng tạo dừng chờ bạn duyệt.
- Notebook này: cài môi trường, đặt kịch bản vào dự án, rồi (tùy chọn) gọi Claude Code ở chế độ không tương tác. Cờ của Claude Code đánh dấu `# TODO: kiểm tra` vì chưa xác minh trên Colab.
- README ghi mặc định: ngưỡng xin duyệt $0.50/hành động, trần ngân sách $10. Không có khóa API vẫn dựng được (Piper TTS offline, nguồn video mở, Remotion/HyperFrames + FFmpeg).
- AGPL-3.0: nếu bạn chạy bản sửa đổi như một dịch vụ mạng thì có nghĩa vụ công bố mã nguồn. README không nói gì về điều khoản của dịch vụ bên thứ ba hay bản quyền nội dung tạo ra; hãy tự kiểm tra.

> **Bước 0:** vào `Runtime > Change runtime type > T4 GPU` rồi bấm Save. Phiên Colab miễn phí có giới hạn thời gian; mọi kết quả đều được lưu ra Google Drive để không mất khi phiên bị ngắt.

Cấu trúc thư mục trên Drive (tự tạo ở ô mount):

```
MyDrive/content-factory/
  input/    <- đưa kịch bản .txt, video, audio lên đây
  output/   <- kết quả để tải về Local
  models/   <- trọng số / checkpoint
```

## 1. Cấu hình (chỉnh ở đây)

In [ ]:
#@title Tham số
PROJECT_NAME = "my-video"      #@param {type:"string"}
SCRIPT_FILE  = "kichban.txt"   #@param {type:"string"}
OPENMONTAGE_COMMIT = ""        #@param {type:"string"}
# Để trống = lấy bản mới nhất. Dán mã commit để ghim phiên bản.
RUN_CLAUDE_CODE = False        #@param {type:"boolean"}
# Cần khóa ANTHROPIC_API_KEY trong Colab Secrets (biểu tượng chìa khóa bên trái).
EXTRA_INSTRUCTION = "Make a vertical 1080x1920 short from this script. Use free assets only."  #@param {type:"string"}

## 2. Mount Drive và kiểm tra GPU

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/content-factory'
IN_DIR, OUT_DIR, MODEL_DIR = f'{BASE}/input', f'{BASE}/output', f'{BASE}/models'
for d in (IN_DIR, OUT_DIR, MODEL_DIR):
    os.makedirs(d, exist_ok=True)
print('Input :', IN_DIR)
print('Output:', OUT_DIR)
print('Models:', MODEL_DIR)

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'KHONG THAY GPU. Hay chon Runtime > Change runtime type > T4 GPU.')

## 3. Cài đặt OpenMontage
Theo README: `git clone`, `make setup` (hoặc `pip install -r requirements.txt` + `npm install` trong `remotion-composer`). Yêu cầu Python 3.10+, FFmpeg, Node 18+ (HyperFrames cần Node 22+).

In [ ]:
import subprocess, os, shutil

def sh(cmd, cwd=None, check=True):
    print('$', cmd)
    r = subprocess.run(cmd, shell=True, cwd=cwd, text=True, capture_output=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        print('LOI:', r.stderr[-3000:])
        if check:
            raise RuntimeError(f'Lenh that bai: {cmd}')
    return r

sh('apt-get -y -qq install ffmpeg make')
print(sh('node -v', check=False).stdout)
# TODO: kiem tra - neu Node < 22, cai Node 22 (vd. qua nodesource) de HyperFrames chay duoc.
if not os.path.isdir('/content/OpenMontage'):
    sh('git clone https://github.com/calesthio/OpenMontage.git /content/OpenMontage')
if OPENMONTAGE_COMMIT:
    sh(f'git checkout {OPENMONTAGE_COMMIT}', cwd='/content/OpenMontage')
sh('git rev-parse HEAD', cwd='/content/OpenMontage')
sh('make setup', cwd='/content/OpenMontage', check=False)  # TODO: kiem tra - neu loi, chay chuoi lenh thay the trong README

## 4. Đưa kịch bản từ Drive/input vào dự án

In [ ]:
import pathlib
src = pathlib.Path(IN_DIR) / SCRIPT_FILE
assert src.exists(), f'Khong thay {src}. Hay dua kich ban .txt len Drive/input.'
script_text = src.read_text(encoding='utf-8')
proj = pathlib.Path('/content/OpenMontage/projects') / PROJECT_NAME
proj.mkdir(parents=True, exist_ok=True)
(proj / 'script.txt').write_text(script_text, encoding='utf-8')
print('Da chep kich ban:', proj / 'script.txt', f'({len(script_text)} ky tu)')
PROMPT = (f"{EXTRA_INSTRUCTION}\nProject name: {PROJECT_NAME}.\n"
          f"The approved script is in projects/{PROJECT_NAME}/script.txt. Render to projects/{PROJECT_NAME}/renders/final.mp4.")
print(PROMPT)

## 5. Chạy dựng video
**Cách A (khuyến nghị, đúng thiết kế của OpenMontage):** mở repo trong một trợ lý AI và dán `PROMPT` ở trên. Trên Colab thì dùng cách B nếu bạn có khóa API.

**Cách B (thử nghiệm):** cài Claude Code CLI và chạy không tương tác.

In [ ]:
import os, subprocess
if not RUN_CLAUDE_CODE:
    print('RUN_CLAUDE_CODE = False: bo qua. Hay dung Cach A, hoac bat tham so nay.')
else:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    subprocess.run('npm install -g @anthropic-ai/claude-code', shell=True, check=True)
    # TODO: kiem tra - co va quyen cua Claude Code khi chay bang root tren Colab.
    r = subprocess.run(['claude', '-p', PROMPT, '--permission-mode', 'acceptEdits'],
                       cwd='/content/OpenMontage', text=True, capture_output=True)
    print(r.stdout[-6000:])
    if r.returncode != 0:
        print('LOI:', r.stderr[-3000:])

## 6. Chép kết quả ra Drive/output

In [ ]:
import glob, shutil, os
found = glob.glob(f'/content/OpenMontage/projects/{PROJECT_NAME}/renders/*.mp4')
if not found:
    print('Chua co video trong renders/. Kiem tra buoc 5 (README chi neu ro duong dan projects/<ten>/renders/final.mp4).')
for f in found:
    dst = os.path.join(OUT_DIR, f'{PROJECT_NAME}_{os.path.basename(f)}')
    shutil.copy2(f, dst)
    print('KET QUA:', dst, f'({os.path.getsize(dst)/1e6:.1f} MB)')